# 05 · Sensors & SLAs

> **Module 2 · Reliability & Operations · Lecture 2.1–2.3**

In this notebook you will:

- Understand what a sensor really is: a **task that waits**, and the two ways it can wait
- Compare `mode="poke"` and `mode="reschedule"` by their worker-slot economics
- Learn the difference between `timeout`, `execution_timeout`, `retries` and an **SLA**
- Measure an SLA against the **data interval end**, not wall-clock now
- Build an `sla_miss_callback` and an SLA compliance tracker

## 1. A sensor is a task that waits

A sensor is not "a task that checks something". It is **a task whose job is to
consume time until a condition holds**. Everything interesting about sensors
follows from that one sentence.

In Airflow there are two families:

- **Python-defined sensors** — `PythonSensor`, `SqlSensor`, `GCSObjectSensor`.
  You supply a callable that returns `True` (ready) or `False` (not yet).
- **Deferrable sensors** — `GCSObjectSensor(deferrable=True)`.
  The worker slot is *released* while the wait happens, and the scheduler
  re-creates the task when the event fires.

And two waiting strategies, for *any* sensor:

| mode | what happens while waiting | worker slot |
|---|---|---|
| `poke` | the worker sleeps and re-checks every `poke_interval` | held for the whole wait |
| `reschedule` | the task exits, is rescheduled, re-checks later | released between checks |

`poke` is simpler and lower latency. `reschedule` is the only choice when the
wait is long (minutes to hours), because a `poke` wait **occupies a worker
slot for its entire duration** — and worker slots are the scarcest resource
in any Airflow deployment, Composer included.

## 2. The rule of thumb

- Wait under ~5 minutes → `poke` is fine.
- Wait longer → `reschedule`, or make it deferrable.
- Wait hours → neither; the condition probably belongs in the *schedule*, not
  in a sensor.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import importlib.util

for name in ("airflow", "google.cloud.bigquery"):
    found = importlib.util.find_spec(name) is not None
    print(f"{name:<28} importable: {found}")

print()
print("This notebook models sensor semantics in plain Python.")
print("No Airflow import is required, and none is attempted.")

## 3. The condition, separated from the waiting

The hard part of a sensor is never the waiting — it is the **condition**. The
waiting is Airflow's job. The condition is *your* logic, and logic belongs in
a module you can unit-test in milliseconds, with no worker, no timeout and no
3 a.m. debugging session.

This course ships that condition in `project/core/sensor_conditions.py`:

```python
def source_is_ready(
    lines: list[str],
    *,
    expected_rows: int = 0,
    reject_threshold: int = 0,
) -> bool:
```

It transforms the raw lines and answers a two-part question:

1. Are there **at least** `expected_rows` accepted rows?
2. Are there **at most** `reject_threshold` rejected rows?

Both bounds matter. A file with 5 000 rows where 4 000 are garbage is
*present* but *not ready*, and a sensor that only checks "does the file
exist" will happily load garbage.

Note that the condition calls `transform_orders` with its default
`statuses=REVENUE_STATUSES` — so it measures the *accepted* count, the same
number the pipeline will actually produce.

In [ ]:
from sensor_conditions import source_is_ready

from pathlib import Path

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent

SAMPLE_DIR = COURSE_ROOT / "sample_data"

orders = (SAMPLE_DIR / "orders.csv").read_text(encoding="utf-8").splitlines()
malformed = (SAMPLE_DIR / "malformed_orders.csv").read_text(encoding="utf-8").splitlines()

print(f"orders.csv lines          : {len(orders)}")
print(f"malformed_orders.csv lines: {len(malformed)}")
print()

# The readiness condition, evaluated against real data.
print("source_is_ready(orders, expected_rows=1245, reject_threshold=0) ->",
      source_is_ready(orders, expected_rows=1245, reject_threshold=0))
print("source_is_ready(orders, expected_rows=2000, reject_threshold=0) ->",
      source_is_ready(orders, expected_rows=2000, reject_threshold=0))
print("source_is_ready(malformed, expected_rows=0, reject_threshold=0) ->",
      source_is_ready(malformed, expected_rows=0, reject_threshold=0))
print("source_is_ready(malformed, expected_rows=0, reject_threshold=1) ->",
      source_is_ready(malformed, expected_rows=0, reject_threshold=1))

## 4. Simulating the two modes

Here is the whole difference between `poke` and `reschedule`, in executable
form. We model a condition that becomes true after `N` polls.

```python
def run_poke(check, polls, interval):
    # one worker slot, occupied for the entire wait
    for attempt in range(1, polls + 1):
        if check():
            return attempt, "held slot throughout"
        time.sleep(interval)
    return polls, "held slot throughout"

def run_reschedule(check, polls, interval):
    # slot released between attempts
    for attempt in range(1, polls + 1):
        if check():
            return attempt, "released slot between polls"
        time.sleep(interval)
    return polls, "released slot between polls"
```

The number of *polls* is identical. The difference is what the scheduler
does with the worker while the answer is "not yet".

The economics, for a Composer environment with `n` worker slots:

| wait | mode | slots occupied | other DAGs can run? |
|---|---|---|---|
| 2 min | `poke` | 1, for 2 min | yes, with `n-1` slots |
| 60 min | `poke` | 1, for 60 min | yes, with `n-1` slots |
| 60 min | `reschedule` | 0, between polls | yes, with all `n` slots |

`reschedule` has a cost: each poll is a scheduler round trip and a new task
instance, so it is *less* responsive and *more* chatty in the metadata
database. That trade-off is worth it precisely when the wait is long.

In [ ]:
import time


def simulate(check, *, polls: int, interval: float, mode: str):
    """Simulate a sensor waiting for a condition to become true.

    Args:
        check: A callable returning True once the condition holds.
        polls: The maximum number of attempts.
        interval: Seconds between polls.
        mode: ``"poke"`` or ``"reschedule"``.

    Returns:
        A ``(polls_used, slot_held_fraction)`` tuple.
    """
    slot_held = 0.0
    for attempt in range(1, polls + 1):
        if mode == "poke":
            slot_held += 1.0          # the worker is occupied this whole period
        if check():
            return attempt, slot_held
        if mode == "reschedule":
            pass                       # no slot held while waiting
        time.sleep(interval)
    return polls, slot_held


ready_after = 5
state = {"polls": 0}


def condition():
    state["polls"] += 1
    return state["polls"] >= ready_after


poke_polls, poke_slots = simulate(condition, polls=10, interval=0.01, mode="poke")
print(f"poke      : {poke_polls} polls, worker slots consumed: {poke_slots:.0f}")

state["polls"] = 0
resched_polls, resched_slots = simulate(condition, polls=10, interval=0.01, mode="reschedule")
print(f"reschedule: {resched_polls} polls, worker slots consumed: {resched_slots:.0f}")

print()
print(f"same polls: {poke_polls == resched_polls}")
print(f"slots saved by reschedule: {poke_slots - resched_slots:.0f}")
print()
print("A 'poke' sensor holds a worker slot for every poll it makes.")
print("That is why long waits use 'reschedule' (or deferrable=True).")

## 5. Timeout vs execution timeout vs retries vs SLA

These four are constantly confused. They are not interchangeable.

| mechanism | question it answers | what happens when it trips | measured against |
|---|---|---|---|
| `timeout` (sensor) | "how long do I wait for the condition?" | the sensor **fails** | now |
| `execution_timeout` | "how long may this task run?" | the task is **killed** | now (task start) |
| `retries` | "how many times may I try again?" | another attempt | n/a |
| `sla` | "by when must this *pipeline* be done?" | `sla_miss_callback` fires | **the data interval end** |

The SLA row is the one that surprises people. An SLA is a **business
contract** ("yesterday's orders are summarised by 06:00"), so it is anchored
to the data, not to the clock. A DAG run for logical date `2026-01-15`
covers the interval `[2026-01-15, 2026-01-16)`, and the SLA deadline is that
interval's **end** plus the SLA duration.

If the SLA were measured from "now", a run that starts at 23:59 and finishes
at 00:01 would be judged against a moving target, and a backfilled run from
last month would be instantly "late" — which is meaningless.

## 6. The ordering that matters

```
source ready?  ──sensor timeout──► task fails
      │
      ▼
transform      ──execution_timeout──► task killed
      │
      ▼
load           ──retries exhausted──► task fails, DAG fails
      │
      ▼
DAG complete   ──past SLA deadline──► sla_miss_callback fires
```

The SLA does not stop anything. It **observes**. The pipeline may still
succeed after missing its SLA; the callback is how you tell the business.

In [ ]:
from datetime import datetime, timedelta, timezone

from sla_logic import sla_deadline, sla_met

SLA = timedelta(hours=2)
LOGICAL_DATE = "2026-01-15"

deadline = sla_deadline(LOGICAL_DATE, SLA)
print(f"logical date      : {LOGICAL_DATE}")
print(f"data interval end : 2026-01-16 00:00:00 UTC")
print(f"SLA duration      : {SLA}")
print(f"deadline          : {deadline}")
print()

candidates = [
    ("on time, finished 01:30", datetime(2026, 1, 16, 1, 30, tzinfo=timezone.utc)),
    ("exactly at deadline", datetime(2026, 1, 16, 2, 0, 0, tzinfo=timezone.utc)),
    ("one minute late", datetime(2026, 1, 16, 2, 1, 0, tzinfo=timezone.utc)),
    ("three hours late", datetime(2026, 1, 16, 3, 0, 0, tzinfo=timezone.utc)),
]

for label, finished_at in candidates:
    met = sla_met(LOGICAL_DATE, SLA, finished_at)
    margin = deadline - finished_at
    print(f"  {label:<28} met={str(met):<5} margin={margin}")

## 7. Why the deadline is the interval end, not "now"

Consider a DAG that runs daily at midnight, processing the previous day's
orders.

- The run with logical date `2026-01-15` processes the interval
  `[2026-01-15, 2026-01-16)` — data that only *exists* after `2026-01-16`
  00:00 UTC.
- With a two-hour SLA, the deadline is `2026-01-16 02:00 UTC`.
- A run that started at `2026-01-16 00:05` and finished at `2026-01-16
  01:30` is **on time**, even though it finished in the small hours.

Now consider a backfill. Airflow re-runs logical dates from the past. Those
runs finish *now*, and they would be trivially "late" against a wall-clock
SLA — punishing you for repairing history. Anchoring to the interval end
means a backfilled run is judged against the same deadline it would have
faced originally, which is the only sensible thing.

> **Rule:** the SLA is a property of the data, not of the scheduler.

## 8. The naive wall-clock comparison, side by side

The code below shows the difference explicitly: the same run can be "on time"
under the interval-end rule and "late" under a wall-clock rule.

In [ ]:
from datetime import datetime, timedelta, timezone

from sla_logic import sla_deadline, sla_met

SLA = timedelta(hours=2)
LOGICAL_DATE = "2026-01-15"
INTERVAL_END = datetime(2026, 1, 16, 0, 0, tzinfo=timezone.utc)

# A run for the 2026-01-15 interval that executes just after midnight.
finished_at = datetime(2026, 1, 16, 0, 30, tzinfo=timezone.utc)

interval_rule = sla_met(LOGICAL_DATE, SLA, finished_at)

# The wrong rule: "the deadline is two hours after the task started".
wrong_deadline = finished_at + SLA
wall_clock_rule = finished_at <= wrong_deadline

print(f"interval-end deadline : {sla_deadline(LOGICAL_DATE, SLA)}")
print(f"wall-clock deadline   : {wrong_deadline}  (measured from finish)")
print()
print(f"interval-end rule says on time : {interval_rule}")
print(f"wall-clock rule says on time   : {wall_clock_rule}")
print()
print("Same run, two rules, two answers - which is exactly why the")
print("SLA must be anchored to the data interval, not to the clock.")

## 9. The SLA miss callback

When the deadline passes, Airflow invokes `sla_miss_callback`. Three
properties matter, and all three are easy to get wrong:

1. **It runs in the scheduler process**, not in a worker. It must be fast and
   must not import heavy libraries or make slow network calls.
2. **It receives the context dict**, so it can read `dag_id`, `task_id`,
   `logical_date` and the data interval.
3. **It must not raise.** An exception in the callback is logged, not
   retried; the most you get is a log line, and the most you lose is the
   notification.

The safe shape is therefore: gather a small payload, hand it to a queue or
a logging call, and return. Paging a human from inside a scheduler callback
is how you turn one missed SLA into a stuck scheduler.

## 10. A callback you can actually test

Below is a callback implemented as a plain function — no Airflow import
required — so it can be unit-tested. The Airflow wiring is one line:

```python
task = PythonOperator(
    task_id="load",
    sla=timedelta(hours=2),
    sla_miss_callback=notify_sla_miss,
)
```

In [ ]:
from datetime import datetime, timedelta, timezone

SLA = timedelta(hours=2)

# Collected instead of sent, so the notebook has no side effects.
NOTIFICATIONS: list[dict] = []


def notify_sla_miss(context) -> str:
    """Build and record an SLA-miss notification.

    Args:
        context: The Airflow task context (or a stand-in with the same keys).

    Returns:
        A one-line summary suitable for a log line.
    """
    payload = {
        "dag_id": context.get("dag_id", "<unknown>"),
        "task_id": context.get("task_id", "<unknown>"),
        "logical_date": str(context.get("logical_date", "<unknown>")),
        "data_interval_end": str(context.get("data_interval_end", "<unknown>")),
        "sla": str(SLA),
        "notified_at": datetime.now(timezone.utc).isoformat(),
    }
    NOTIFICATIONS.append(payload)
    # In production this is where you'd enqueue an email/Slack/pager call.
    # It must be fast and must never raise.
    return (
        f"SLA MISS dag={payload['dag_id']} task={payload['task_id']} "
        f"logical_date={payload['logical_date']}"
    )


fake_context = {
    "dag_id": "etl_to_gcs",
    "task_id": "load",
    "logical_date": "2026-01-15",
    "data_interval_end": "2026-01-16T00:00:00+00:00",
}

print(notify_sla_miss(fake_context))
print()
print("Recorded payload:")
for key, value in NOTIFICATIONS[0].items():
    print(f"  {key:<18} {value}")

## 11. One tracker, many runs

A single SLA miss is an incident; a *rate* of SLA misses is a quality
metric. `project/core/sla_tracker.py` records the outcome of each run and
computes compliance over a period.

This is deliberately outside Airflow: Airflow emits the events, and you
aggregate them over days or weeks to check the contractual number. A tracker
that lives inside a task can only ever see one run.

In [ ]:
from datetime import datetime, timedelta, timezone

from sla_tracker import SlaTracker

SLA = timedelta(hours=2)

tracker = SlaTracker(SLA)

# A week of runs. The first four finish on time, the last two do not.
finished_times = [
    datetime(2026, 1, 15, 23, 40, tzinfo=timezone.utc),   # on time
    datetime(2026, 1, 16, 23, 50, tzinfo=timezone.utc),   # on time
    datetime(2026, 1, 17, 23, 30, tzinfo=timezone.utc),   # on time
    datetime(2026, 1, 18, 23, 45, tzinfo=timezone.utc),   # on time
    datetime(2026, 1, 19, 23, 55, tzinfo=timezone.utc),   # on time
    datetime(2026, 1, 20, 5, 10, tzinfo=timezone.utc),    # LATE
    datetime(2026, 1, 21, 4, 0, tzinfo=timezone.utc),     # LATE
]

logical_dates = [
    "2026-01-15", "2026-01-16", "2026-01-17", "2026-01-18",
    "2026-01-19", "2026-01-20", "2026-01-21",
]

for logical_date, finished_at in zip(logical_dates, finished_times):
    met = tracker.record(logical_date, finished_at)
    print(f"{logical_date}  finished={finished_at.strftime('%H:%M')}  met={met}")

print()
print(tracker.report())
print()
print(f"Compliance target 99% met: {tracker.compliance() >= 0.99}")

## 12. Sensor design checklist

Before you write a sensor, answer these:

1. **Is the condition pure?** If it needs the database, the network, or the
   clock to decide, it belongs in a function you can test.
2. **How long will the wait realistically be?** Under 5 minutes → `poke`.
   Longer → `reschedule`, or deferrable.
3. **What happens on timeout?** A sensor timeout is a task failure, and it
   burns every retry. Decide the retry count deliberately.
4. **Is `soft_fail` appropriate?** `soft_fail=True` turns the timeout into a
   *skip* rather than a failure — correct only when the rest of the DAG has
   a legitimate "this data doesn't exist" branch.
5. **What is the readiness threshold?** "File exists" is almost never the
   condition you mean. Use row counts, size thresholds, or a sentinel.
6. **Does the condition need to be idempotent?** Sensors are re-checked
   every `poke_interval`, so the check runs many times.

## Exercises

1. **Timeout versus SLA.** For a daily DAG with a two-hour SLA and a sensor that waits up to 30 minutes, give the `timeout`, `execution_timeout`, `retries` and `sla` values you would use, and say which one fires if the source file never arrives.
2. **Harder — a readiness rule.** Write a function `ready_for_load(lines)` that returns True only when the data has at least 1 000 accepted rows **and** fewer than 100 rejected rows. Test it against `orders.csv` and against `malformed_orders.csv`, and explain why a plain "file exists" check would be wrong for both.
3. **Harder — the backfill SLA.** A run for logical date `2026-01-15` is executed by a backfill on `2026-01-20` and finishes at `2026-01-20 04:00 UTC`. Using a two-hour SLA, decide whether the run met its SLA, and explain why the answer must not depend on the date the backfill was executed.

<details>
<summary><b>Solution (click to reveal)</b></summary>

**1.** `timeout=timedelta(minutes=30)` on the sensor, `execution_timeout=timedelta(minutes=45)` on the transform task, `retries=3` with `retry_delay=timedelta(minutes=5)`, `sla=timedelta(hours=2)` on the DAG's final task. If the source never arrives, the **sensor timeout** fires first — after 30 minutes — and the task fails; that failure then consumes one of the three retries.

**2.**

```python
def ready_for_load(lines):
    accepted, rejected = transform_orders(lines)
    return len(accepted) >= 1000 and len(rejected) < 100
```

Against `orders.csv` this returns True (1 245 accepted, 0 rejected); against `malformed_orders.csv` it returns False (0 accepted, 5 rejected). A "file exists" check is wrong for both: for the malformed file the file *does* exist, so an existence check would pass and the pipeline would load garbage; for the good file an existence check would also pass even if the upload were truncated to a header plus ten rows, because the file is still there.

**3.** The deadline for logical date `2026-01-15` is the interval end `2026-01-16 00:00 UTC` plus two hours, i.e. `2026-01-16 02:00 UTC`. The run finished at `2026-01-20 04:00 UTC`, so it **missed** the SLA. The answer depends only on the logical date and the finish time of the *data interval*, never on when the backfill was executed — otherwise every backfill would be automatically "late" for the crime of repairing history, and the metric would measure operator behaviour instead of pipeline health.

</details>